# 69-Claim CFR+ Random Cap/Fit Screen

Three 20-minute A100-oriented runs to isolate random action cap and regret fit effort after ruling out hash sampling. Each run saves an average-policy snapshot, trains an approximate BR for 5 minutes, evaluates that BR once per minute, and plots the results.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
import gc
import json
from pathlib import Path
import re
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.br_fitted_return_action_conditioned import ActionConditionedFittedReturnBRTrainer
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.eval.approx_br import evaluate_approximate_br
from liars_poker.serialization import load_policy, save_policy

assert torch.cuda.is_available(), 'This screen is intended for a CUDA machine.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')

spec = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)
assert len(rules_for_spec(spec).claims) == 69

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
RUN_ROOT = REPO_ROOT / 'artifacts' / 'cfr_plus_69_claim_random_cap_fit_screen' / f'{spec.to_short_str()}___{RUN_ID}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_MINUTES = 20.0
BR_MINUTES = 5.0
BR_EVALUATE_EVERY_MINUTES = 1.0
PRINT_EVERY_MINUTES = 2.0
TRAVERSAL_BATCH_SIZE = 4096
ACTION_CHUNK_SIZE = 1_048_576
RECORD_FLUSH_SIZE = 262_144

print('repo:', REPO_ROOT)
print('run root:', RUN_ROOT)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))


In [ ]:
def json_default(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, tuple):
        return list(obj)
    raise TypeError(type(obj).__name__)


def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, default=json_default, sort_keys=True) + '\n')


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def slugify(text: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')


def mean_or_nan(value) -> float:
    if value is None:
        return float('nan')
    if isinstance(value, (int, float)):
        return float(value)
    values = list(value)
    return float(np.mean(values)) if values else float('nan')


def gpu_mem_gib() -> tuple[float, float]:
    free, total = torch.cuda.mem_get_info()
    return round(free / 2**30, 2), round(total / 2**30, 2)


In [ ]:
BASE_TRAINER_KWARGS = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'seed': 7,
    'regret_buffer_capacity': 4_000_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'strategy_train_steps': 6,
    'strategy_weighting': 'linear',
    'regret_positive_weight': 0.5,
    'validation_fraction': 0.001,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': TRAVERSAL_BATCH_SIZE,
    'traverser_action_sample_count': None,
    'traverser_action_sample_fraction': None,
    'traverser_action_sample_mode': 'random',
    'traverser_action_baseline': 'none',
    'traversal_streaming': True,
    'traversal_live_row_budget': None,
    'traverser_action_chunk_size': ACTION_CHUNK_SIZE,
    'traversal_record_flush_size': RECORD_FLUSH_SIZE,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
    'device': device,
}

CONFIGS = [
    {
        'label': 'A cap24 trav512 regret24',
        'traversals_per_player': 512,
        'cap_schedule': (24,),
        'regret_train_steps': 24,
    },
    {
        'label': 'B cap24 trav512 regret48',
        'traversals_per_player': 512,
        'cap_schedule': (24,),
        'regret_train_steps': 48,
    },
    {
        'label': 'C cap32 trav512 regret48',
        'traversals_per_player': 512,
        'cap_schedule': (32,),
        'regret_train_steps': 48,
    },
]

manifest = {
    'run_type': 'cfr_plus_69_claim_random_cap_fit_screen',
    'spec': spec.to_json(),
    'train_minutes': TRAIN_MINUTES,
    'br_minutes': BR_MINUTES,
    'base_trainer_kwargs': {k: (str(v) if k == 'device' else v) for k, v in BASE_TRAINER_KWARGS.items()},
    'configs': CONFIGS,
}
(RUN_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=json_default), encoding='utf-8')
pd.DataFrame(CONFIGS)


In [ ]:
def train_config(config: dict) -> tuple[Path, dict]:
    label = config['label']
    slug = slugify(label)
    run_dir = RUN_ROOT / slug
    policy_dir = run_dir / 'policy'
    summary_path = run_dir / 'summary.json'

    if summary_path.exists() and (policy_dir / 'metadata.json').exists():
        print('already trained:', label)
        return policy_dir, json.loads(summary_path.read_text(encoding='utf-8'))

    kwargs = dict(BASE_TRAINER_KWARGS)
    kwargs.update({
        'regret_train_steps': int(config['regret_train_steps']),
        'traverser_action_sample_schedule': tuple(config['cap_schedule']),
    })

    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / 'config.json').write_text(
        json.dumps({'config': config, 'trainer_kwargs': kwargs}, indent=2, default=json_default),
        encoding='utf-8',
    )

    trainer = DeepCFRPlusTrainer(spec, **kwargs)
    measured_s = 0.0
    budget_s = 60.0 * TRAIN_MINUTES
    next_print_s = 0.0
    start_wall = time.perf_counter()

    print(f"\n=== training {label} ===")
    print('free / total GPU GiB:', gpu_mem_gib())
    while measured_s < budget_s:
        iter_start = time.perf_counter()
        record = trainer.run_iteration(traversals_per_player=int(config['traversals_per_player']))
        iter_s = time.perf_counter() - iter_start
        measured_s += iter_s

        timing = record.get('timing', {})
        sampling = record.get('action_sampling', {})
        row = {
            'label': label,
            'iteration': int(trainer.iteration),
            'measured_training_s': float(measured_s),
            'measured_training_min': float(measured_s / 60.0),
            'wall_s': float(time.perf_counter() - start_wall),
            'iteration_s': float(iter_s),
            'traversals_per_player': int(config['traversals_per_player']),
            'cap_schedule': list(config['cap_schedule']),
            'regret_train_steps': int(config['regret_train_steps']),
            'traversal_s': float(timing.get('traversal_s', np.nan)),
            'regret_fit_s': float(timing.get('regret_training_s', np.nan)),
            'strategy_fit_s': float(timing.get('strategy_training_s', np.nan)),
            'claim_edge_fraction': float(sampling.get('claim_edge_fraction', np.nan)),
            'regret_weight_ess_fraction': float(sampling.get('regret_weight_ess_fraction', np.nan)),
            'largest_regret_weight': float(sampling.get('largest_regret_weight', np.nan)),
            'streamed_edge_chunks': int(sampling.get('streamed_edge_chunks', 0)),
            'streamed_row_splits': int(sampling.get('streamed_row_splits', 0)),
            'regret_loss': mean_or_nan(record.get('regret_loss')),
            'strategy_loss': mean_or_nan(record.get('strategy_loss')),
            'new_regret_records': int(sum(record.get('new_regret_records') or [])),
            'new_strategy_records': int(sum(record.get('new_strategy_records') or [])),
        }
        append_jsonl(run_dir / 'training.jsonl', row)

        if measured_s >= next_print_s:
            print(
                f"{label}: train={measured_s/60:.1f}m iter={trainer.iteration} "
                f"iter_s={iter_s:.3f} trav={row['traversal_s']:.3f}s "
                f"fit={row['regret_fit_s']:.3f}/{row['strategy_fit_s']:.3f}s "
                f"edges={row['claim_edge_fraction']:.3f} ESS={row['regret_weight_ess_fraction']:.3f}"
            )
            next_print_s += 60.0 * PRINT_EVERY_MINUTES

    save_policy(trainer.average_policy(), str(policy_dir))
    rows = read_jsonl(run_dir / 'training.jsonl')
    summary = {
        'label': label,
        'policy_dir': str(policy_dir),
        'iterations': int(trainer.iteration),
        'measured_training_min': float(measured_s / 60.0),
        'mean_traversal_s': float(np.mean([r['traversal_s'] for r in rows])),
        'mean_regret_fit_s': float(np.mean([r['regret_fit_s'] for r in rows])),
        'mean_strategy_fit_s': float(np.mean([r['strategy_fit_s'] for r in rows])),
        'mean_claim_edge_fraction': float(np.mean([r['claim_edge_fraction'] for r in rows])),
        'mean_regret_weight_ess_fraction': float(np.nanmean([r['regret_weight_ess_fraction'] for r in rows])),
        'final_policy_dir': str(policy_dir),
    }
    summary_path.write_text(json.dumps(summary, indent=2, default=json_default), encoding='utf-8')
    print('saved policy:', policy_dir)

    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    return policy_dir, summary


In [ ]:
BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'lr': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
    'seed': 17,
    'device': device,
}

# Kept alive across cells/reruns while the kernel survives.
BR_STATES = globals().get('BR_STATES', {})


def get_br_state(label: str, policy_dir: Path) -> dict:
    if label in BR_STATES:
        return BR_STATES[label]

    policy, loaded_spec = load_policy(str(policy_dir))
    if loaded_spec != spec:
        raise ValueError(f'Spec mismatch for {policy_dir}: {loaded_spec}')

    trainer = ActionConditionedFittedReturnBRTrainer(spec, policy, **BR_TRAINER_KWARGS)
    state = {
        'label': label,
        'policy_dir': Path(policy_dir),
        'trainer': trainer,
        'measured_s': 0.0,
        'next_eval_s': 60.0 * BR_EVALUATE_EVERY_MINUTES,
        'out_dir': RUN_ROOT / slugify(label) / 'br',
    }
    state['out_dir'].mkdir(parents=True, exist_ok=True)
    BR_STATES[label] = state
    return state


def continue_br(label: str, policy_dir: Path, *, total_minutes: float = BR_MINUTES) -> pd.DataFrame:
    state = get_br_state(label, policy_dir)
    trainer = state['trainer']
    out_dir = state['out_dir']
    target_s = 60.0 * total_minutes

    print(f"\n=== BR {label} to {total_minutes:.1f}m total ===")
    while state['measured_s'] < target_s:
        start = time.perf_counter()
        record = trainer.run_iteration(episodes_per_role=4096, rollout_batch_size=1024)
        iter_s = time.perf_counter() - start
        state['measured_s'] += iter_s
        train_row = {
            'label': label,
            'iteration': int(trainer.iteration),
            'measured_responder_s': float(state['measured_s']),
            'measured_responder_min': float(state['measured_s'] / 60.0),
            'iteration_s': float(iter_s),
            'record': record,
        }
        append_jsonl(out_dir / 'training.jsonl', train_row)

        if state['measured_s'] >= state['next_eval_s'] or state['measured_s'] >= target_s:
            metrics = evaluate_approximate_br(
                trainer,
                episodes_per_role=200_000,
                rollout_batch_size=8192,
            )
            eval_row = {
                'label': label,
                'policy_dir': str(policy_dir),
                'br_iteration': int(trainer.iteration),
                'responder_training_min': float(state['measured_s'] / 60.0),
                **metrics,
            }
            append_jsonl(out_dir / 'evaluations.jsonl', eval_row)
            append_jsonl(RUN_ROOT / 'br_evaluations.jsonl', eval_row)
            print(
                f"{label}: BR={state['measured_s']/60:.1f}m iter={trainer.iteration} "
                f"estimate={metrics['exploitability_estimate']:.5f} "
                f"LCB={metrics['exploitability_lower_bound']:.5f}"
            )
            while state['next_eval_s'] <= state['measured_s']:
                state['next_eval_s'] += 60.0 * BR_EVALUATE_EVERY_MINUTES

    return pd.DataFrame(read_jsonl(out_dir / 'evaluations.jsonl'))


In [ ]:
# Run all three training configs back to back, with a 5-minute approximate BR after each.
TRAIN_SUMMARIES = []
BR_SUMMARIES = []

for config in CONFIGS:
    policy_dir, train_summary = train_config(config)
    TRAIN_SUMMARIES.append(train_summary)
    br_frame = continue_br(config['label'], policy_dir, total_minutes=BR_MINUTES)
    if not br_frame.empty:
        BR_SUMMARIES.append(br_frame.iloc[-1].to_dict())

train_summary_df = pd.DataFrame(TRAIN_SUMMARIES)
br_summary_df = pd.DataFrame(BR_SUMMARIES)
display(train_summary_df)
display(br_summary_df)


In [ ]:
# If you want to continue BRs later without retraining, rerun this cell with a larger target, e.g. 10, 15, or 20.
CONTINUE_BR_TO_MINUTES = 5.0

policy_dirs = {
    summary['label']: Path(summary['policy_dir'])
    for summary in TRAIN_SUMMARIES
} if 'TRAIN_SUMMARIES' in globals() and TRAIN_SUMMARIES else {
    config['label']: RUN_ROOT / slugify(config['label']) / 'policy'
    for config in CONFIGS
}

for label, policy_dir in policy_dirs.items():
    if (policy_dir / 'metadata.json').exists():
        continue_br(label, policy_dir, total_minutes=CONTINUE_BR_TO_MINUTES)
    else:
        print('missing policy, skipping:', label, policy_dir)


In [ ]:
# Plot training throughput and per-minute BR curves.
training_rows = []
for config in CONFIGS:
    training_rows.extend(read_jsonl(RUN_ROOT / slugify(config['label']) / 'training.jsonl'))
train_df = pd.DataFrame(training_rows)
br_df = pd.DataFrame(read_jsonl(RUN_ROOT / 'br_evaluations.jsonl'))

if not train_df.empty:
    final_train = (
        train_df.sort_values('measured_training_min')
        .groupby('label')
        .tail(1)
        .set_index('label')
    )
    display(final_train[[
        'iteration', 'measured_training_min', 'traversals_per_player',
        'cap_schedule', 'regret_train_steps', 'traversal_s', 'regret_fit_s',
        'strategy_fit_s', 'claim_edge_fraction', 'regret_weight_ess_fraction',
    ]])

if not br_df.empty:
    br_df = br_df.sort_values(['label', 'responder_training_min'])
    final_br = br_df.groupby('label').tail(1).set_index('label')
    display(final_br[[
        'responder_training_min', 'p_first', 'p_second',
        'exploitability_estimate', 'exploitability_lower_bound',
    ]].sort_values('exploitability_estimate'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if not train_df.empty:
    for label, group in train_df.groupby('label'):
        axes[0].plot(group['measured_training_min'], group['iteration'], label=label)
    axes[0].set_title('CFR+ iterations during 20-minute screen')
    axes[0].set_xlabel('Training minutes')
    axes[0].set_ylabel('Iteration')
    axes[0].grid(True, alpha=0.3)

    timing = train_df.groupby('label')[['traversal_s', 'regret_fit_s', 'strategy_fit_s']].mean()
    timing.plot(kind='bar', stacked=True, ax=axes[1])
    axes[1].set_title('Mean iteration cost')
    axes[1].set_ylabel('Seconds')
    axes[1].tick_params(axis='x', rotation=25)

if not br_df.empty:
    for label, group in br_df.groupby('label'):
        axes[2].plot(
            group['responder_training_min'],
            group['exploitability_estimate'],
            marker='o',
            label=label,
        )
        axes[2].plot(
            group['responder_training_min'],
            group['exploitability_lower_bound'],
            linestyle='--',
            alpha=0.55,
        )
    axes[2].set_title('Approximate BR estimate by responder compute')
    axes[2].set_xlabel('Responder training minutes')
    axes[2].set_ylabel('Discovered exploitability')
    axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
